# SQL Business Analysis — UCI Online Retail 

## Overview

This notebook uses SQL (via DuckDB) to investigate customer retention, purchase behavior, and revenue distribution using the UCI Online Retail dataset.

It extends a separate Python-based RFM + KMeans segmentation analysis by quantifying funnel drop-offs and behavioral differences across customer segments. The segmentation results are imported here for cross-analysis.

## Dataset

The dataset contains transactional records from a UK-based online retailer, covering December 2010 – December 2011.

| Field | Description |
|---|---|
| `CustomerID` | Unique customer identifier |
| `InvoiceNo` | Transaction ID (prefix `C` = cancellation) |
| `StockCode` | Product code |
| `Description` | Product name |
| `Quantity` | Units per line item |
| `UnitPrice` | Price per unit (GBP) |
| `InvoiceDate` | Timestamp of transaction |
| `Country` | Customer country |

Each row represents a single product line within a transaction.

## Setup

Connect to DuckDB and load the dataset. Rows with missing `CustomerID`, cancelled orders (InvoiceNo starting with `C`), or non-positive Quantity / UnitPrice are removed before analysis.

In [3]:
import duckdb
import pandas as pd

con = duckdb.connect()

In [4]:
df = pd.read_csv("online_retail.csv")


In [9]:
df.shape

(541909, 8)

In [5]:
df_clean = df.copy()

# remove missing customer
df_clean = df_clean.dropna(subset=['CustomerID'])

# remove cancelled orders
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]

# remove invalid values
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]

df_clean.shape


(397884, 8)

In [6]:
con.register("retail", df_clean)

---

## Case 1: Repeat Purchase Behavior

### Business Question

What percentage of customers return to make a second purchase after their first transaction?

Early-stage retention is typically the largest drop-off point in the customer lifecycle. Understanding this metric reveals whether the business builds repeat demand or relies primarily on one-time buyers.

### Overall Repeat Purchase Rate

In [18]:
query = """
WITH customer_orders AS (
    SELECT 
        CustomerID,
        COUNT(DISTINCT DATE(InvoiceDate)) AS order_count
    FROM retail
    GROUP BY CustomerID
)

SELECT 
    COUNT(*) FILTER (WHERE order_count > 1) * 1.0 
        / COUNT(*) AS repeat_purchase_rate
FROM customer_orders
"""

con.execute(query).fetchdf()

,repeat_purchase_rate
0,0.643154


In [20]:
query = """
WITH first_purchase AS (
    SELECT 
        CustomerID,
        MIN(DATE(InvoiceDate)) AS first_date
    FROM retail
    GROUP BY CustomerID
),

customer_orders AS (
    SELECT 
        r.CustomerID,
        COUNT(DISTINCT DATE(r.InvoiceDate)) AS active_days,
        SUM(CASE WHEN DATE(r.InvoiceDate) > f.first_date THEN 1 ELSE 0 END) AS repeat_activity
    FROM retail r
    JOIN first_purchase f
        ON r.CustomerID = f.CustomerID
    GROUP BY r.CustomerID
)

SELECT 
    COUNT(*) FILTER (WHERE repeat_activity > 0) * 1.0 
        / COUNT(*) AS repeat_purchase_rate
FROM customer_orders


"""

con.execute(query).fetchdf()

,repeat_purchase_rate
0,0.643154


Both approaches confirm a **64.3% overall repeat purchase rate** — a high number, but this is a lifetime metric. The question is *when* customers return.

### Time-Windowed Repeat Rate (30 / 60 / 90 days)

In [21]:
query = """
WITH first_purchase AS (
    SELECT 
        CustomerID,
        MIN(DATE(InvoiceDate)) AS first_date
    FROM retail
    GROUP BY CustomerID
),

purchases AS (
    SELECT 
        r.CustomerID,
        DATE(r.InvoiceDate) AS purchase_date,
        f.first_date
    FROM retail r
    JOIN first_purchase f
        ON r.CustomerID = f.CustomerID
),

flags AS (
    SELECT 
        CustomerID,

        MAX(CASE 
            WHEN purchase_date > first_date 
             AND purchase_date <= first_date + INTERVAL 30 DAY 
            THEN 1 ELSE 0 END) AS repeat_30d,

        MAX(CASE 
            WHEN purchase_date > first_date 
             AND purchase_date <= first_date + INTERVAL 60 DAY 
            THEN 1 ELSE 0 END) AS repeat_60d,

        MAX(CASE 
            WHEN purchase_date > first_date 
             AND purchase_date <= first_date + INTERVAL 90 DAY 
            THEN 1 ELSE 0 END) AS repeat_90d

    FROM purchases
    GROUP BY CustomerID
)

SELECT 
    COUNT(*) FILTER (WHERE repeat_30d = 1) * 1.0 / COUNT(*) AS repeat_30d_rate,
    COUNT(*) FILTER (WHERE repeat_60d = 1) * 1.0 / COUNT(*) AS repeat_60d_rate,
    COUNT(*) FILTER (WHERE repeat_90d = 1) * 1.0 / COUNT(*) AS repeat_90d_rate
FROM flags
"""
con.execute(query).fetchdf()

,repeat_30d_rate,repeat_60d_rate,repeat_90d_rate
0,0.187183,0.335639,0.422084


| Window | Rate | Incremental Gain |
|--------|------|-----------------|
| 0 – 30 days | 18.7% | baseline |
| 0 – 60 days | 33.6% | **+14.9 pp** ← largest jump |
| 0 – 90 days | 42.2% | +8.6 pp |

The sharpest gain occurs in the **31–60 day window**, nearly double the gain of the 60–90 day window. A large segment of customers naturally returns here but likely needs a nudge.

### Inter-Purchase Gap (Validation)

In [12]:
query = """
WITH order_dates AS (
    SELECT
        CAST(CustomerID AS INTEGER) AS CustomerID,
        DATE(InvoiceDate) AS purchase_date,
        LAG(DATE(InvoiceDate)) OVER (
            PARTITION BY CAST(CustomerID AS INTEGER) 
            ORDER BY DATE(InvoiceDate)
        ) AS prev_date
    FROM retail
    WHERE CustomerID IS NOT NULL
      AND Quantity > 0
      AND UnitPrice > 0
    GROUP BY CustomerID, DATE(InvoiceDate)
)
SELECT
    ROUND(AVG(DATEDIFF('day', prev_date, purchase_date)), 1) AS avg_days_between_purchases,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (
        ORDER BY DATEDIFF('day', prev_date, purchase_date)
    ), 1) AS median_days,
    MIN(DATEDIFF('day', prev_date, purchase_date)) AS min_days,
    MAX(DATEDIFF('day', prev_date, purchase_date)) AS max_days
FROM order_dates
WHERE prev_date IS NOT NULL
"""

con.execute(query).fetchdf()

,avg_days_between_purchases,median_days,min_days,max_days
0,45.7,28.0,1,366


The distribution is right-skewed: median gap = **28 days**, mean = **45.7 days**. This confirms a segment of slow returners and directly validates the 45-day coupon window below.

### Purchase Frequency Distribution

In [9]:
query = """
WITH customer_orders AS (
    SELECT 
        CustomerID,
        COUNT(DISTINCT DATE(InvoiceDate)) AS active_days
    FROM retail
    GROUP BY CustomerID
)

SELECT 
    CASE 
        WHEN active_days = 1 THEN '1 purchase'
        WHEN active_days = 2 THEN '2 purchases'
        WHEN active_days <= 5 THEN '3-5 purchases'
        ELSE '6+ purchases'
    END AS purchase_bucket,
    COUNT(*) AS customers
FROM customer_orders
GROUP BY purchase_bucket
ORDER BY purchase_bucket
"""
con.execute(query).fetchdf()

,purchase_bucket,customers
0,1 purchase,1548
1,2 purchases,874
2,3-5 purchases,1117
3,6+ purchases,799


### 🎯 Key Recommendation: 45-Day Post-First-Purchase Coupon

The biggest retention gap is the **1→2 purchase transition** (34.4% of customers never return). Once customers make a second purchase, retention stabilizes at ~70–75%.

The data suggests a 45-day coupon window:

```
Day 0       Day 28 (median)   Day 45 (coupon expiry)   Day 60 (return peak)
|-----------|-----------------|------------------------|
  issued      typical return    urgency deadline          natural peak
```

**Why not 30 days?** Expires before the high-conversion window opens.  
**Why not 90 days?** Removes urgency — customers procrastinate.  
**45 days** overlaps the peak return window (day 31–60) while preserving a deadline.

**Implementation:**
1. Trigger coupon 3 days after first purchase
2. Set expiry to 45 days from first purchase date
3. Offer 10–15% discount — enough to motivate, not enough to erode margin
4. Track redemption rate vs. control group to validate lift in 60-day retention

---

## Case 2: Purchase Funnel (First-to-Nth Conversion)

### Business Question

At each successive purchase, what share of customers continue? Where is the sharpest drop-off?

In [16]:
query = """
WITH order_sequence AS (
    SELECT
        CAST(CustomerID AS INTEGER) AS CustomerID,
        InvoiceNo,
        DATE(InvoiceDate) AS purchase_date,
        ROW_NUMBER() OVER (
            PARTITION BY CAST(CustomerID AS INTEGER) 
            ORDER BY DATE(InvoiceDate)
        ) AS purchase_rank
    FROM retail
    WHERE CustomerID IS NOT NULL 
      AND Quantity > 0
    GROUP BY CustomerID, InvoiceNo, DATE(InvoiceDate)
)
SELECT
    COUNT(DISTINCT CASE WHEN purchase_rank >= 1 THEN CustomerID END) AS made_1st,
    COUNT(DISTINCT CASE WHEN purchase_rank >= 2 THEN CustomerID END) AS made_2nd,
    COUNT(DISTINCT CASE WHEN purchase_rank >= 3 THEN CustomerID END) AS made_3rd,
    COUNT(DISTINCT CASE WHEN purchase_rank >= 4 THEN CustomerID END) AS made_4th,
    COUNT(DISTINCT CASE WHEN purchase_rank >= 5 THEN CustomerID END) AS made_5th,

    ROUND(COUNT(DISTINCT CASE WHEN purchase_rank >= 2 THEN CustomerID END) * 100.0 /
          COUNT(DISTINCT CASE WHEN purchase_rank >= 1 THEN CustomerID END), 1) AS "1→2_%",
    ROUND(COUNT(DISTINCT CASE WHEN purchase_rank >= 3 THEN CustomerID END) * 100.0 /
          COUNT(DISTINCT CASE WHEN purchase_rank >= 2 THEN CustomerID END), 1) AS "2→3_%",
    ROUND(COUNT(DISTINCT CASE WHEN purchase_rank >= 4 THEN CustomerID END) * 100.0 /
          COUNT(DISTINCT CASE WHEN purchase_rank >= 3 THEN CustomerID END), 1) AS "3→4_%",
    ROUND(COUNT(DISTINCT CASE WHEN purchase_rank >= 5 THEN CustomerID END) * 100.0 /
          COUNT(DISTINCT CASE WHEN purchase_rank >= 4 THEN CustomerID END), 1) AS "4→5_%"
FROM order_sequence
"""

con.execute(query).fetchdf()

,made_1st,made_2nd,made_3rd,made_4th,made_5th,1→2_%,2→3_%,3→4_%,4→5_%
0,4338,2845,2010,1502,1114,65.6,70.7,74.7,74.2


| Step | Customers | Conversion |
|------|-----------|-----------|
| 1st purchase | 4,338 | — |
| 2nd purchase | 2,845 | **65.6%** ← critical drop |
| 3rd purchase | 2,010 | 70.7% |
| 4th purchase | 1,502 | 74.7% |
| 5th purchase | 1,114 | 74.2% |

**34.4% of customers never make a second purchase.** Once past the 1→2 step, retention stabilizes in the 70–75% range. This identifies the 1→2 transition as the single highest-leverage intervention point — directly addressed by the 45-day coupon in Case 1.

---

## Case 3: Behavioral Analysis by Customer Segment

### Business Question

How do the RFM-derived customer segments differ in order frequency, revenue, and purchase cadence? Which segments should be prioritized for which strategies?

Load RFM cluster labels from the segmentation notebook and register them in DuckDB.

In [15]:
# Load RFM cluster labels from the segmentation notebook
# Register in DuckDB
con.execute("CREATE TABLE rfm_clusters AS SELECT * FROM read_csv_auto('rfm_clusters.csv')")

# Verify
con.execute("SELECT * FROM rfm_clusters LIMIT 5").fetchdf()

,CustomerID,Cluster
0,12346.0,3
1,12347.0,0
2,12348.0,0
3,12349.0,0
4,12350.0,1


### Cluster Behavior Summary

In [17]:
query = """
WITH retail_stats AS (
    SELECT
        CAST(CustomerID AS INTEGER) AS CustomerID,
        COUNT(DISTINCT InvoiceNo) AS num_orders,
        ROUND(SUM(Quantity * UnitPrice), 2) AS total_revenue,
        ROUND(SUM(Quantity * UnitPrice) / COUNT(DISTINCT InvoiceNo), 2) AS avg_order_value,
        COUNT(DISTINCT StockCode) AS unique_products,
        COUNT(DISTINCT DATE(InvoiceDate)) AS active_days
    FROM retail
    WHERE Quantity > 0
      AND UnitPrice > 0
      AND CustomerID IS NOT NULL
    GROUP BY CustomerID
),
clusters AS (
    SELECT
        CAST(CAST(CustomerID AS FLOAT) AS INTEGER) AS CustomerID,
        Cluster
    FROM rfm_clusters
)
SELECT
    c.Cluster,
    COUNT(*) AS num_customers,
    ROUND(AVG(r.total_revenue), 2) AS avg_total_revenue,
    ROUND(AVG(r.num_orders), 2) AS avg_orders,
    ROUND(AVG(r.avg_order_value), 2) AS avg_order_value,
    ROUND(AVG(r.unique_products), 2) AS avg_unique_products,
    ROUND(AVG(r.active_days), 2) AS avg_active_days
FROM clusters c
JOIN retail_stats r ON c.CustomerID = r.CustomerID
GROUP BY c.Cluster
ORDER BY avg_total_revenue DESC
"""

con.execute(query).fetchdf()

,Cluster,num_customers,avg_total_revenue,avg_orders,avg_order_value,avg_unique_products,avg_active_days
0,2,13,127338.31,82.54,8570.73,670.23,56.85
1,3,204,12709.09,22.33,1080.41,184.99,19.00
2,0,3054,1359.05,3.68,376.76,63.05,3.46
3,1,1067,480.62,1.55,314.81,26.05,1.47


| Cluster | Size | Avg Revenue | Avg Orders | Avg AOV | Interpretation |
|---------|------|-------------|------------|---------|----------------|
| 2 | 13 | £127,338 | 82.5 | £8,571 | **Whales** — account managers, any churn is critical |
| 3 | 204 | £12,709 | 22.3 | £1,080 | **High-value, low-frequency** — prime target for 45-day coupon |
| 0 | 3,054 | £1,359 | 3.7 | £377 | **Mid-tier** — increase purchase frequency via bundles |
| 1 | 1,067 | £481 | 1.6 | £315 | **At-risk / lapsed** — win-back or deprioritize |

**Cluster 3 is the highest-ROI coupon target:** high AOV (£1,080 vs £377 for mid-tier) means even a modest reactivation rate returns significant revenue per coupon issued.

## Summary

This analysis followed a single thread: **where do customers drop off, and what can be done about it?**

### Retention Funnel (Case 1 & 2)

64% of customers make a repeat purchase at some point, but the **1→2 transition is the critical failure point** — 34.4% of customers never return after their first purchase. Once past this step, retention stabilizes at ~70–75%. Time-windowed analysis shows the sharpest recovery window is day 31–60, with an inter-purchase mean of 45.7 days.

> **Recommendation:** A 45-day post-first-purchase coupon targets this window directly — avoiding the premature 30-day expiry while preserving urgency that a 90-day window removes.

### Segment Targeting (Case 3)

| Cluster | Size | AOV | Strategy |
|---------|------|-----|----------|
| 2 | 13 | £8,571 | Dedicated account management — any churn is critical |
| 3 | 204 | £1,080 | Prime coupon target — highest reactivation ROI |
| 0 | 3,054 | £377 | Increase frequency via bundles |
| 1 | 1,067 | £315 | Evaluate win-back cost vs. expected LTV |

### Limitations & Next Steps

- Dataset covers a single year; seasonal effects are not isolated
- Coupon recommendation is data-informed but untested — A/B test needed to validate lift in 60-day retention
- Country-level segmentation not explored; majority UK but may reveal behavioral differences